# 08 — Advanced Fidelity Tier 1

GMM fitting, adversarial distinguishability testing (AUC ≈ 0.5 = indistinguishable), temporal autocorrelation, and FFT periodicity detection.

In [ ]:
from sqllocks_spindle import Spindle
from sqllocks_spindle.inference import AdvancedProfiler
import importlib, pandas as pd, numpy as np
from sqllocks_spindle.cli import _get_domain_registry

spindle = Spindle()
reg = _get_domain_registry()
mod, cls, _ = reg['retail']
domain = getattr(importlib.import_module(mod), cls)(schema_mode='3nf')
real = spindle.generate(domain=domain, scale='small', seed=42)
synth = spindle.generate(domain=domain, scale='small', seed=99)

In [ ]:
# Run advanced profiling
profiler = AdvancedProfiler(max_gmm_components=3)

# Use the first shared table
table = next(t for t in real.tables if t in synth.tables)
adv = profiler.profile_pair(real.tables[table], synth.tables[table], table_name=table)
print(f'Table: {table}')

In [ ]:
# GMM fits
print('\n--- GMM Component Fits ---')
for col, fit in adv.gmm_fits.items():
    print(f'  {col}: {fit.n_components} components  BIC={fit.bic:.1f}')
    for i, (m, w, s) in enumerate(zip(fit.means, fit.weights, fit.stds)):
        print(f'    component {i}: mean={m:.1f}  weight={w:.3f}  std={s:.1f}')

In [ ]:
# Adversarial test
if adv.adversarial:
    auc = adv.adversarial.auc_roc
    passed = adv.adversarial.passed
    score = adv.adversarial.distinguishability_score
    print(f'\n--- Adversarial Distinguishability ---')
    print(f'  AUC-ROC: {auc:.3f}  (0.5 = perfect, 1.0 = distinguishable)')
    print(f'  Distinguishability score: {score:.1f}/100 (lower is better)')
    print(f'  Passed (AUC < 0.75): {passed}')
    print(f'  Most distinguishing features:')
    for feat, imp in adv.adversarial.top_features[:5]:
        print(f'    {feat}: {imp:.4f}')

In [ ]:
# Temporal + FFT periodicity
print('\n--- Temporal Profiles ---')
for col, tp in adv.temporal_profiles.items():
    print(f'  {col}: mean gap={tp.mean_gap_seconds:.0f}s  autocorr_lag1={tp.autocorrelation_lag1:.3f}')

print('\n--- FFT Periodicity ---')
for col, pr in adv.periodicity.items():
    periodic = 'PERIODIC' if pr.is_periodic else 'not periodic'
    print(f'  {col}: {periodic}  dominant_period={pr.dominant_period}')